# BART Training Pipeline

## Purpose
Full BART v1 (standard fine-tune) + v2 (plan-guided) training, validation, and inference tuning on Cochrane-auto dataset.

## Execution Order
1. Mount Google Drive, install dependencies
2. Load train (11,510) + val (1,472) sentence pairs
3. Train label classifiers: DistilBERT -> RoBERTa (best: 52.6%) -> DeBERTa
4. Train BART v1 (epoch 1-3, standard fine-tune)
5. Train BART v2 (epoch 4-5, plan-guided with special tokens)
6. Grid search over inference hyperparams (num_beams, length_penalty, ngram)
7. Evaluate: v1 SARI=26.37, v2 oracle SARI=35.86, v2 realistic SARI=33.23

## Outputs
- BART v1 checkpoint
- BART v2 checkpoint  
- RoBERTa label classifier

## Hardware
NVIDIA T4 GPU (16GB VRAM), Google Colab

## Notes
- Classifier evolution: DistilBERT (initial) -> RoBERTa (final) -> DeBERTa (exploratory)
- RoBERTa accuracy: 52.6% (flat, best of 5 epochs on 1,472 val samples)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: Mountpoint must not already contain files

In [ ]:
!pip install transformers datasets sentencepiece -q
!pip install git+https://github.com/feralvam/easse.git

  Cloning https://github.com/feralvam/easse.git to /tmp/pip-req-build-12gs807h
  Running command git clone --filter=blob:none --quiet https://github.com/feralvam/easse.git /tmp/pip-req-build-12gs807h
  Resolved https://github.com/feralvam/easse.git to commit 6a4352ec299ed03fda8ee45445ca43d9c7673e89
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/text-simplification-evaluation.git (to revision main) to /tmp/pip-install-zgcapl1t/tseval_04e6340170764b53b9873f05a3e1d5bd
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/text-simplification-evaluation.git /tmp/pip-install-zgcapl1t/tseval_04e6340170764b53b9873f05a3e1d5bd
  Resolved https://github.com/facebookresearch/text-simplification-evaluation.git to commit dea8863683ea5946fd50184883c9be7a7339e821
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.0 MB/s eta 0:00:00
  

In [ ]:
from easse.sari import corpus_sari

# Quick test with dummy data
orig  = ["The cat sat on the mat."]
sys   = ["The cat sat on the mat."]
refs  = [["The cat was on the mat."]]

score = corpus_sari(orig_sents=orig, sys_sents=sys, refs_sents=refs)
print(f"✅ EASSE working! Test SARI: {score:.2f}")

✅ EASSE working! Test SARI: 22.45


In [ ]:
import os

# Create a project folder in your Drive
project_path = '/content/drive/MyDrive/SimpleText2025'
os.makedirs(project_path, exist_ok=True)

# Clone cochrane-auto repo into Drive
%cd /content/drive/MyDrive/SimpleText2025
!git clone https://github.com/JanB100/cochrane-auto

/content/drive/MyDrive/SimpleText2025
Cloning into 'cochrane-auto'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 45 (delta 15), reused 40 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 12.11 MiB | 23.70 MiB/s, done.
Resolving deltas: 100% (15/15), done.


```

Your folder structure will look like:
```

MyDrive/

  └── SimpleText2025/

    └── cochrane-auto/
        └── data/
            ├── cochraneauto_sents_train.csv
            ├── cochraneauto_sents_val.csv
            ├── cochraneauto_sents_test.csv
            ├── cochraneauto_docs_train.csv
            ├── cochraneauto_docs_val.csv
            └── cochraneauto_docs_test.csv

In [ ]:
import pandas as pd
import ast

data_path = '/content/drive/MyDrive/SimpleText2025/cochrane-auto/data'

# Task 1.1
train_11 = pd.read_csv(f'{data_path}/cochraneauto_sents_train.csv')
val_11   = pd.read_csv(f'{data_path}/cochraneauto_sents_val.csv')
test_11  = pd.read_csv(f'{data_path}/cochraneauto_sents_test.csv')

print("Train size:", len(train_11))
print(train_11[['complex', 'simple', 'label']].head(3))

Train size: 11510
                                             complex  \
0  Three studies (146 participants) met our selec...   
1  Two studies compared multidisciplinary, fast-t...   
2  Two were RCTs with parallel design (total 94 p...   

                                              simple     label  
0  ['Three studies involving 146 participants wer...  rephrase  
1                                                 []    delete  
2                                                 []    delete  


In [ ]:
# Task 1.2
train_12 = pd.read_csv(f'{data_path}/cochraneauto_docs_train.csv')
val_12   = pd.read_csv(f'{data_path}/cochraneauto_docs_val.csv')

print("Train size:", len(train_12))
print(train_12[['complex', 'simple']].head(2))

Train size: 849
                                             complex  \
0  Three studies (146 participants) met our selec...   
1  We included five trials, in which 1406 infants...   

                                              simple  
0  Three studies involving 146 participants were ...  
1  We found five studies that involved 1406 babie...  


In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration
import torch

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

def simplify_sentence(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=200,
            min_length=20,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test it
sample = train_11['complex'].iloc[0]
print("ORIGINAL:\n", sample)
print("\nSIMPLIFIED:\n", simplify_sentence(sample))

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

ORIGINAL:
 Three studies (146 participants) met our selection criteria.

SIMPLIFIED:
 Three studies (146 participants) met our selection criteria. Three studies met our criteria.


In [ ]:
# Quick check on a few val examples
for i in range(3):
    row = val_11.iloc[i]
    print(f"--- Example {i+1} ---")
    print("COMPLEX:", row['complex'])
    print("PREDICTED:", simplify_sentence(row['complex']))
    import ast
    try:
        ref = ' '.join(ast.literal_eval(row['simple']))
    except:
        ref = str(row['simple'])
    print("REFERENCE:", ref)
    print()

--- Example 1 ---
COMPLEX: We included 16 RCTs (2232 couples analysed).
PREDICTED: We included 16 RCTs (2232 couples analysed) and 16 couples analysed.
REFERENCE: We found 16 randomised controlled trials (studies where treatments are decided at random; these usually give the most reliable evidence about treatment effects) comparing glucocorticoids around the time of embryo implantation versus no glucocorticoids or placebo (dummy treatment), in 2232 couples undergoing IVF/ICSI.

--- Example 2 ---
COMPLEX: We are uncertain whether glucocorticoids improved live birth rates (odds ratio (OR) 1.37, 95% confidence interval (CI) 0.69 to 2.71; 2 RCTs, n = 366; I2 = 7%; very low-certainty evidence).
PREDICTED: We are uncertain whether glucocorticoids improved live birth rates (odds ratio (OR) 1.37, 95% confidence interval (CI) 0.69 to 2.71; 2 RCTs, n = 366)
REFERENCE: Considering the quality of evidence, we are uncertain whether there was a difference in live birth rates after glucocorticoids.



Fine-tuning Code

Cell 1 — Prepare Dataset Class


In [ ]:
import ast
import torch
from torch.utils.data import Dataset

class CochraneDataset(Dataset):
    def __init__(self, df, tokenizer, max_input_len=512, max_target_len=256):
        self.data = df.dropna(subset=['complex', 'simple']).reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.data)

    def parse_simple(self, s):
        try:
            return ' '.join(ast.literal_eval(s))
        except:
            return str(s)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        complex_text = str(row['complex'])
        simple_text  = self.parse_simple(row['simple'])

        # Tokenize input
        model_inputs = self.tokenizer(
            complex_text,
            max_length=self.max_input_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )

        # ✅ Fixed: tokenize target directly, no as_target_tokenizer
        label_encodings = self.tokenizer(
            text_target=simple_text,        # ← key fix
            max_length=self.max_target_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )

        # Replace pad token id with -100 so loss ignores padding
        label_ids = label_encodings['input_ids'].squeeze()
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids':      model_inputs['input_ids'].squeeze(),
            'attention_mask': model_inputs['attention_mask'].squeeze(),
            'labels':         label_ids
        }

Cell 2 — Create Datasets


In [ ]:
train_dataset = CochraneDataset(train_11, tokenizer)
val_dataset   = CochraneDataset(val_11,   tokenizer)

print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")

# Sanity check
sample_item = train_dataset[0]
print("Input shape :", sample_item['input_ids'].shape)
print("Label shape :", sample_item['labels'].shape)
print("\nSample input decoded:")
print(tokenizer.decode(sample_item['input_ids'], skip_special_tokens=True)[:100])

Train samples : 11510
Val samples   : 1697
Input shape : torch.Size([512])
Label shape : torch.Size([256])

Sample input decoded:
Three studies (146 participants) met our selection criteria.


Cell 3 — Set Up Training Arguments


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import os

model_save_path = '/content/drive/MyDrive/SimpleText2025/models/bart-cochrane-11'
os.makedirs(model_save_path, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=model_save_path,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=True,
    generation_max_length=256,
    report_to='none'
    # ✅ removed logging_dir — was deprecated
)

print("Training args set up successfully!")

Training args set up successfully!


Cell 4 — Train


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(    # ✅ removed the 'N' prefix
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("🚀 Starting fine-tuning...")
trainer.train()
print("✅ Done!")

🚀 Starting fine-tuning...


Epoch,Training Loss,Validation Loss
1,1.672743,1.800807
2,1.179334,1.882465
3,0.839740,2.032137


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


✅ Done!


Cell 5 — Save Fine-tuned Model to Drive

In [ ]:
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✅ Model saved to {model_save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /content/drive/MyDrive/SimpleText2025/models/bart-cochrane-11


Cell 6 — Reload Model (Run this if Colab disconnects)


In [ ]:
# Only run this if your session restarted
# Otherwise skip and continue from Cell 7

from transformers import BartTokenizer, BartForConditionalGeneration
import torch

model_save_path = '/content/drive/MyDrive/SimpleText2025/models/bart-cochrane-11'

tokenizer = BartTokenizer.from_pretrained(model_save_path)
model = BartForConditionalGeneration.from_pretrained(model_save_path)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Model reloaded from Drive")

Cell 7 — Updated simplify_sentence() Function


In [ ]:
def simplify_sentence(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256,
            min_length=20,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3    # ✅ avoids repetition
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Cell 8 — Compare Before vs After Fine-tuning on Val Examples


In [ ]:
import ast

print("=" * 60)
print("TESTING FINE-TUNED MODEL ON VALIDATION EXAMPLES")
print("=" * 60)

for i in range(5):
    row = val_11.iloc[i]

    try:
        ref = ' '.join(ast.literal_eval(row['simple']))
    except:
        ref = str(row['simple'])

    predicted = simplify_sentence(row['complex'])

    print(f"\n--- Example {i+1} ---")
    print(f"COMPLEX  : {row['complex']}")
    print(f"PREDICTED: {predicted}")
    print(f"REFERENCE: {ref}")
    print(f"LABEL    : {row['label']}")

TESTING FINE-TUNED MODEL ON VALIDATION EXAMPLES

--- Example 1 ---
COMPLEX  : We included 16 RCTs (2232 couples analysed).
PREDICTED: We included 16 studies (2232 couples) in this review. The studies were conducted in Africa, Asia, Latin America, and Asia.
REFERENCE: We found 16 randomised controlled trials (studies where treatments are decided at random; these usually give the most reliable evidence about treatment effects) comparing glucocorticoids around the time of embryo implantation versus no glucocorticoids or placebo (dummy treatment), in 2232 couples undergoing IVF/ICSI.
LABEL    : rephrase

--- Example 2 ---
COMPLEX  : We are uncertain whether glucocorticoids improved live birth rates (odds ratio (OR) 1.37, 95% confidence interval (CI) 0.69 to 2.71; 2 RCTs, n = 366; I2 = 7%; very low-certainty evidence).
PREDICTED: We are uncertain whether glucocorticoids improved live birth rates (2 studies, 366 women; very low-certainty evidence).
REFERENCE: Considering the quality of evide

Cell 10 — Evaluate SARI Score on Validation Set


In [ ]:
!pip install git+https://github.com/feralvam/easse.git -q
print("✅ EASSE installed")

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.1 MB/s eta 0:00:00
✅ EASSE installed


In [ ]:
from easse.sari import corpus_sari

# Use first 200 val samples to save time
val_sample = val_11.dropna(subset=['complex', 'simple']).head(200).reset_index(drop=True)

orig_sents = val_sample['complex'].tolist()

# Parse reference simplifications
ref_sents = []
for s in val_sample['simple']:
    try:
        ref_sents.append(' '.join(ast.literal_eval(s)))
    except:
        ref_sents.append(str(s))

# Generate predictions
print("Generating predictions... (takes ~3-5 mins)")
sys_sents = []
for i, text in enumerate(orig_sents):
    sys_sents.append(simplify_sentence(text))
    if (i+1) % 50 == 0:
        print(f"  Done {i+1}/{len(orig_sents)}")

# Compute SARI
sari = corpus_sari(
    orig_sents=orig_sents,
    sys_sents=sys_sents,
    refs_sents=[ref_sents]
)

print(f"\n{'='*40}")
print(f"SARI Score (fine-tuned BART) : {sari:.2f}")
print(f"Target from paper            : ~35.5")
print(f"Baseline (no fine-tune)      : ~15-20")
print(f"{'='*40}")

Generating predictions... (takes ~3-5 mins)
  Done 50/200
  Done 100/200
  Done 150/200
  Done 200/200

SARI Score (fine-tuned BART) : 26.37
Target from paper            : ~35.5
Baseline (no fine-tune)      : ~15-20


Cell 11 — Generate Task 1.1 Submission File


In [ ]:
import os
import pandas as pd

output_path = '/content/drive/MyDrive/SimpleText2025/outputs'
os.makedirs(output_path, exist_ok=True)

# Load the official test file
test_11 = pd.read_json(
    '/content/drive/MyDrive/SimpleText2025/simpletext25_task11_test.json'
)

print(f"Test samples: {len(test_11)}")
print(test_11.head(2))

Test samples: 9160
    pair_id  para_id  sent_id  \
0  CD012520        0        0   
1  CD012520        0        1   

                                             complex  
0  We included seven cluster-randomised trials wi...  
1  Health professional participants (numbers not ...  


Cell 12 — Run Predictions on Test Set


In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq
import torch

def simplify_batch(texts, batch_size=32):
    all_predictions = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        # Tokenize entire batch at once
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            max_length=512,
            truncation=True,
            padding=True          # pad to longest in batch
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=256,
                min_length=10,
                num_beams=2,          # ✅ reduced from 4 to 2 for speed
                early_stopping=True,
                no_repeat_ngram_size=3
            )

        # Decode all outputs in batch
        decoded = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )
        all_predictions.extend(decoded)

        if (i + batch_size) % 500 == 0 or i == 0:
            print(f"  Done {min(i+batch_size, len(texts))}/{len(texts)}")

    return all_predictions

# Run predictions
texts = test_11['complex'].astype(str).tolist()
print(f"🚀 Running batch predictions on {len(texts)} samples...")
print("Expected time: ~10-15 mins with batch_size=32")

predictions = simplify_batch(texts, batch_size=32)

test_11['prediction'] = predictions
test_11['run_id'] = 'tokatrons_task11_BARTFineTuned'

print(f"\n✅ Predictions done!")
print(test_11[['pair_id', 'complex', 'prediction']].head(3))

🚀 Running batch predictions on 9160 samples...
Expected time: ~10-15 mins with batch_size=32
  Done 32/9160
  Done 4000/9160
  Done 8000/9160

✅ Predictions done!
    pair_id                                            complex  \
0  CD012520  We included seven cluster-randomised trials wi...   
1  CD012520  Health professional participants (numbers not ...   
2  CD012520  Interventions in all studies included implemen...   

                                          prediction  
0  We found seven studies with 42,489 participant...  
1  Health professional participants (numbers not ...  
2  Interventions in all studies included strategi...  


Cell 13 — Save Task 1.1 Submission File

In [ ]:
submission_path = f'{output_path}/tokatrons_task11_BARTFineTuned.json'
test_11.to_json(submission_path, orient='records')
print(f"✅ Submission saved to {submission_path}")

# Verify format is correct
verify = pd.read_json(submission_path)
print("\nColumns:", verify.columns.tolist())
print("Required: ['pair_id', 'para_id', 'sent_id', 'complex', 'prediction', 'run_id']")
print("Shape:", verify.shape)

✅ Submission saved to /content/drive/MyDrive/SimpleText2025/outputs/tokatrons_task11_BARTFineTuned.json

Columns: ['pair_id', 'para_id', 'sent_id', 'complex', 'prediction', 'run_id']
Required: ['pair_id', 'para_id', 'sent_id', 'complex', 'prediction', 'run_id']
Shape: (9160, 6)


Cell 14 — Zip and Prepare for CodaBench Upload


In [ ]:
import zipfile
import os

zip_path = f'{output_path}/tokatrons_task11_BARTFineTuned.zip'

with zipfile.ZipFile(zip_path, 'w') as zf:
    # ✅ File must be at ROOT of zip, not inside a folder
    zf.write(
        submission_path,
        arcname='tokatrons_task11_BARTFineTuned.json'
    )

print(f"✅ Zipped submission at: {zip_path}")
print("Ready to upload to https://www.codabench.org/competitions/8400/")


✅ Zipped submission at: /content/drive/MyDrive/SimpleText2025/outputs/tokatrons_task11_BARTFineTuned.zip
Ready to upload to https://www.codabench.org/competitions/8400/


In [ ]:
import shutil
from google.colab import files

# Zip entire outputs folder
shutil.make_archive(
    '/content/outputs_all',           # output zip name
    'zip',                            # format
    '/content/drive/MyDrive/SimpleText2025/outputs'  # folder to zip
)

print("✅ Zipped!")

# Download it
files.download('/content/outputs_all.zip')

✅ Zipped!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


---

## Current Progress
```
✅ Data downloaded
✅ Environment set up
✅ Data loaded
✅ Baseline model loaded
✅ Fine-tuning complete
✅ Model saved to Drive
⬜ SARI evaluation (Cell 10)
⬜ Generate submission JSON (Cell 12-13)
⬜ Zip and upload to CodaBench (Cell 14)
```

##NEW APPROACH

In [ ]:
# ============================================================
# CELL 1 — Mount Drive + Install
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets sentencepiece -q
!pip install git+https://github.com/feralvam/easse.git -q

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.2 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 2 — Load Data
# ============================================================
import pandas as pd
import ast
import os

data_path = '/content/drive/MyDrive/SimpleText2025/cochrane-auto/data'

train_df = pd.read_csv(f'{data_path}/cochraneauto_sents_train.csv')
val_df   = pd.read_csv(f'{data_path}/cochraneauto_sents_val.csv')

# Parse the 'simple' column from list-string to actual list
def parse_simple(s):
    try:
        lst = ast.literal_eval(s)
        return ' '.join(lst) if isinstance(lst, list) else str(lst)
    except:
        return str(s)

train_df = train_df.dropna(subset=['complex', 'simple', 'label'])
val_df   = val_df.dropna(subset=['complex', 'simple', 'label'])

train_df['simple_text'] = train_df['simple'].apply(parse_simple)
val_df['simple_text']   = val_df['simple'].apply(parse_simple)

# Normalize label → uppercase control token
VALID_LABELS = {'rephrase', 'delete', 'split', 'merge', 'copy'}
train_df['label'] = train_df['label'].str.strip().str.lower()
val_df['label']   = val_df['label'].str.strip().str.lower()
train_df = train_df[train_df['label'].isin(VALID_LABELS)]
val_df   = val_df[val_df['label'].isin(VALID_LABELS)]

# Build plan-guided input: "[REPHRASE] <complex sentence>"
train_df['guided_input'] = '[' + train_df['label'].str.upper() + '] ' + train_df['complex']
val_df['guided_input']   = '[' + val_df['label'].str.upper()   + '] ' + val_df['complex']

print(f"Train: {len(train_df)} | Val: {len(val_df)}")
print(train_df[['guided_input', 'simple_text']].head(3))

Train: 10188 | Val: 1472
                                        guided_input  \
0  [REPHRASE] Three studies (146 participants) me...   
1  [DELETE] Two studies compared multidisciplinar...   
2  [DELETE] Two were RCTs with parallel design (t...   

                                         simple_text  
0  Three studies involving 146 participants were ...  
1                                                     
2                                                     


In [ ]:
# ============================================================
# CELL 3 (FIXED) — Load from checkpoint subfolder
# ============================================================
import os
from transformers import BartTokenizer, BartForConditionalGeneration
import torch

model_save_path = '/content/drive/MyDrive/SimpleText2025/models/bart-cochrane-11'
new_model_path  = '/content/drive/MyDrive/SimpleText2025/models/bart-plan-guided'

# Find the best checkpoint (highest number = last epoch)
checkpoints = [d for d in os.listdir(model_save_path) if d.startswith('checkpoint')]
checkpoints.sort(key=lambda x: int(x.split('-')[-1]))

print("Found checkpoints:")
for c in checkpoints:
    print(f"  {c}")

best_ckpt = os.path.join(model_save_path, checkpoints[-1])
print(f"\n✅ Loading from: {best_ckpt}")

tokenizer = BartTokenizer.from_pretrained(best_ckpt)
model     = BartForConditionalGeneration.from_pretrained(best_ckpt)

# Add control tokens
special_tokens = ['[REPHRASE]', '[DELETE]', '[SPLIT]', '[MERGE]', '[COPY]']
tokenizer.add_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = model.to(device)
print(f"Model on: {device}")
print(f"Vocab size after control tokens: {len(tokenizer)}")

Found checkpoints:
  checkpoint-2878

✅ Loading from: /content/drive/MyDrive/SimpleText2025/models/bart-cochrane-11/checkpoint-2878


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model on: cuda
Vocab size after control tokens: 50270


In [ ]:
# ============================================================
# CELL 4 — Dataset Class
# ============================================================
from torch.utils.data import Dataset

class GuidedCochraneDataset(Dataset):
    def __init__(self, df, tokenizer, max_input=256, max_target=128):
        self.inputs  = df['guided_input'].tolist()
        self.targets = df['simple_text'].tolist()
        self.tokenizer = tokenizer
        self.max_input  = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.inputs[idx],
            max_length=self.max_input,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        with self.tokenizer.as_target_tokenizer() if hasattr(self.tokenizer, 'as_target_tokenizer') else __import__('contextlib').nullcontext():
            dec = self.tokenizer(
                text_target=self.targets[idx],
                max_length=self.max_target,
                truncation=True,
                padding='max_length',
                return_tensors='pt'
            )
        labels = dec['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         labels
        }

train_dataset = GuidedCochraneDataset(train_df, tokenizer)
val_dataset   = GuidedCochraneDataset(val_df,   tokenizer)
print(f"Train dataset: {len(train_dataset)} | Val dataset: {len(val_dataset)}")

Train dataset: 10188 | Val dataset: 1472


In [ ]:
# ============================================================
# CELL 5 — Fine-tune (Plan-Guided)
#
# Strategy to hit ~35.5 SARI within 1-2 hours:
#   - Start from your 26.37 checkpoint (not scratch)
#   - 2 epochs with guided inputs
#   - Estimated time: ~45-60 min on T4
# ============================================================
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=new_model_path,
    num_train_epochs=2,           # 2 epochs on top of existing checkpoint
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=200,
    weight_decay=0.01,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,
    report_to='none',
    logging_steps=100,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(new_model_path)
tokenizer.save_pretrained(new_model_path)
print(f"Plan-guided model saved to: {new_model_path}")

Epoch,Training Loss,Validation Loss
1,1.250014,2.033663
2,0.884821,2.151012


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Plan-guided model saved to: /content/drive/MyDrive/SimpleText2025/models/bart-plan-guided


In [ ]:
# ============================================================
# CELL 6 — Evaluate SARI on Validation Set
# ============================================================
from easse.sari import corpus_sari

model.eval()

def simplify_batch(texts, batch_size=64):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', max_length=256,
                        truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model.generate(**enc, max_length=128, num_beams=2, do_sample=False)
        all_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return all_preds

# Use GROUND TRUTH labels for guided eval (oracle mode — upper bound)
guided_inputs = val_df['guided_input'].tolist()
predictions   = simplify_batch(guided_inputs)

orig_sentences = val_df['complex'].tolist()
references     = [[r] for r in val_df['simple_text'].tolist()]

sari = corpus_sari(
    orig_sents=orig_sentences,
    sys_sents=predictions,
    refs_sents=list(zip(*references))
)
print(f"\n✅ SARI Score (oracle labels): {sari:.2f}")
print(f"   Target: 35.5 | Gap: {35.5 - sari:.2f}")


✅ SARI Score (oracle labels): 31.02
   Target: 35.5 | Gap: 4.48


In [ ]:
# ============================================================
# CELL 7 — Train a Label Classifier (needed for real test data
#           where you don't have ground truth labels)
#           ~10-15 min to train, lightweight
# ============================================================
from transformers import (DistilBertTokenizerFast,
                          DistilBertForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset as HFDataset
import numpy as np

label2id = {'rephrase': 0, 'delete': 1, 'split': 2, 'merge': 3, 'copy': 4}
id2label = {v: k for k, v in label2id.items()}

clf_tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
clf_model     = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

def tokenize_clf(batch):
    return clf_tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_clf_data = HFDataset.from_dict({
    'text':  train_df['complex'].tolist(),
    'label': [label2id[l] for l in train_df['label'].tolist()]
})
val_clf_data = HFDataset.from_dict({
    'text':  val_df['complex'].tolist(),
    'label': [label2id[l] for l in val_df['label'].tolist()]
})

train_clf_tok = train_clf_data.map(tokenize_clf, batched=True)
val_clf_tok   = val_clf_data.map(tokenize_clf, batched=True)

clf_args = TrainingArguments(
    output_dir='/content/label_clf',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
    fp16=True,
)

clf_trainer = Trainer(
    model=clf_model,
    args=clf_args,
    train_dataset=train_clf_tok,
    eval_dataset=val_clf_tok,
)
clf_trainer.train()

clf_model_path = '/content/drive/MyDrive/SimpleText2025/models/label-classifier'
clf_model.save_pretrained(clf_model_path)
clf_tokenizer.save_pretrained(clf_model_path)
print("Label classifier saved!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Label classifier saved!


## RERUNNING FOR CELL8 EXECUTION

In [ ]:
# RELOAD CELL A — Mount + Imports
from google.colab import drive
drive.mount('/content/drive')

import torch, os, json, zipfile
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# RELOAD CELL B — Load fine-tuned BART
from transformers import BartTokenizer, BartForConditionalGeneration

model_path = '/content/drive/MyDrive/SimpleText2025/models/bart-plan-guided'
tokenizer  = BartTokenizer.from_pretrained(model_path)
model      = BartForConditionalGeneration.from_pretrained(model_path).to(device)
print(f"BART loaded on: {device}")

def simplify_batch(texts, batch_size=64):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', max_length=256,
                        truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model.generate(**enc, max_length=128, num_beams=1, do_sample=False)
        all_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return all_preds



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:01<?, ?it/s]

BART loaded on: cuda


In [ ]:
# RELOAD CELL C — Load label classifier
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

clf_path      = '/content/drive/MyDrive/SimpleText2025/models/label-classifier'
clf_tokenizer = DistilBertTokenizerFast.from_pretrained(clf_path)
clf_model     = DistilBertForSequenceClassification.from_pretrained(clf_path).to(device)

id2label = {0:'rephrase', 1:'delete', 2:'split', 3:'merge', 4:'copy'}

print("✅ Label classifier loaded!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Label classifier loaded!


In [ ]:
# ============================================================
# CELL 8 — Generate Submission File for Task 1.1 Test Set
# ============================================================
import json, os, zipfile

test_path = '/content/drive/MyDrive/SimpleText2025/simpletext25_task11_test.json'
with open(test_path) as f:
    test_data = json.load(f)

test_sentences = [item['complex'] for item in test_data]

# Step 1: Predict labels
clf_model.eval()
def predict_labels(texts, batch_size=128):
    all_labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = clf_tokenizer(batch, return_tensors='pt', truncation=True,
                            padding=True, max_length=128).to(device)
        clf_model_gpu = clf_model.to(device)
        with torch.no_grad():
            logits = clf_model_gpu(**enc).logits
        preds = logits.argmax(dim=-1).cpu().tolist()
        all_labels.extend([id2label[p] for p in preds])
    return all_labels

predicted_labels = predict_labels(test_sentences)

# Step 2: Build guided inputs and simplify
guided_test_inputs = [f'[{lbl.upper()}] {sent}'
                      for lbl, sent in zip(predicted_labels, test_sentences)]
test_predictions   = simplify_batch(guided_test_inputs)

# Step 3: Build submission JSON
output = []
for item, pred in zip(test_data, test_predictions):
    output.append({
        'pair_id':    item['pair_id'],
        'para_id':    item['para_id'],
        'sent_id':    item['sent_id'],
        'complex':    item['complex'],
        'prediction': pred,
        'run_id':     'tokatrons_task11_BARTPlanGuided'
    })

# ✅ FIX: create the outputs folder if it doesn't exist
out_path = '/content/drive/MyDrive/SimpleText2025/outputs/tokatrons_task11_BARTPlanGuided.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)

with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

zip_path = out_path.replace('.json', '.zip')
with zipfile.ZipFile(zip_path, 'w') as zf:
    zf.write(out_path, arcname='tokatrons_task11_BARTPlanGuided.json')

print(f"✅ Submission ready: {zip_path}")
print(f"   Total predictions: {len(output)}")

The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Submission ready: /content/drive/MyDrive/SimpleText2025/outputs/tokatrons_task11_BARTPlanGuided.zip
   Total predictions: 9160


In [ ]:
##plan to cross 35.5

In [ ]:
# ============================================================
# CELL 1 — Mount + Install
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


!pip install transformers datasets sentencepiece -q
!pip install git+https://github.com/feralvam/easse.git -q

Mounted at /content/drive
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.0 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 2 — Load Data
# ============================================================
import pandas as pd, ast, os, torch

data_path = '/content/drive/MyDrive/SimpleText2025/cochrane-auto/data'
train_df  = pd.read_csv(f'{data_path}/cochraneauto_sents_train.csv')
val_df    = pd.read_csv(f'{data_path}/cochraneauto_sents_val.csv')

def parse_simple(s):
    try:
        lst = ast.literal_eval(s)
        return ' '.join(lst) if isinstance(lst, list) else str(lst)
    except:
        return str(s)

VALID_LABELS = {'rephrase', 'delete', 'split', 'merge', 'copy'}

for df in [train_df, val_df]:
    df.dropna(subset=['complex', 'simple', 'label'], inplace=True)
    df['simple_text']   = df['simple'].apply(parse_simple)
    df['label']         = df['label'].str.strip().str.lower()

train_df = train_df[train_df['label'].isin(VALID_LABELS)].reset_index(drop=True)
val_df   = val_df[val_df['label'].isin(VALID_LABELS)].reset_index(drop=True)

train_df['guided_input'] = '[' + train_df['label'].str.upper() + '] ' + train_df['complex']
val_df['guided_input']   = '[' + val_df['label'].str.upper()   + '] ' + val_df['complex']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Device: {device}")

Train: 10188 | Val: 1472 | Device: cuda


In [ ]:
# ============================================================
# CELL 3 — Train RoBERTa Label Classifier (~15 min)
#           Replaces DistilBERT — more accurate labels on test set
#           Anti-overfitting: early stopping + label smoothing
# ============================================================
from transformers import (RobertaTokenizerFast, RobertaForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from datasets import Dataset as HFDataset
import numpy as np

label2id = {'rephrase': 0, 'delete': 1, 'split': 2, 'merge': 3, 'copy': 4}
id2label  = {v: k for k, v in label2id.items()}

clf_tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base')
clf_model     = RobertaForSequenceClassification.from_pretrained(
    'roberta-base', num_labels=5, id2label=id2label, label2id=label2id
)

def tokenize_clf(batch):
    return clf_tokenizer(batch['text'], truncation=True,
                         padding='max_length', max_length=128)

train_clf = HFDataset.from_dict({
    'text':  train_df['complex'].tolist(),
    'label': [label2id[l] for l in train_df['label']]
}).map(tokenize_clf, batched=True)

val_clf = HFDataset.from_dict({
    'text':  val_df['complex'].tolist(),
    'label': [label2id[l] for l in val_df['label']]
}).map(tokenize_clf, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': (preds == labels).mean()}

clf_args = TrainingArguments(
    output_dir='/content/roberta_clf',
    num_train_epochs=5,                  # enough to converge
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,                   # L2 regularisation — prevents overfitting
    label_smoothing_factor=0.1,          # prevents overconfident predictions
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none',
    fp16=True,
)

clf_trainer = Trainer(
    model=clf_model,
    args=clf_args,
    train_dataset=train_clf,
    eval_dataset=val_clf,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # stops if val acc drops
)

clf_trainer.train()

clf_path = '/content/drive/MyDrive/SimpleText2025/models/roberta-label-clf'
clf_model.save_pretrained(clf_path)
clf_tokenizer.save_pretrained(clf_path)
print("✅ RoBERTa classifier saved!")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/10188 [00:00<?, ? examples/s]

Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ RoBERTa classifier saved!


In [ ]:
# ============================================================
# CELL 4 — Load v2 BART + Resume Training for 2 More Epochs
#           Anti-overfitting: lower LR, warmup, early stopping
#           Estimated time: ~45 min on T4
# ============================================================
from transformers import (BartTokenizer, BartForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          EarlyStoppingCallback)
from torch.utils.data import Dataset

v2_path = '/content/drive/MyDrive/SimpleText2025/models/bart-plan-guided'
v3_path = '/content/drive/MyDrive/SimpleText2025/models/bart-plan-guided-v3'

tokenizer = BartTokenizer.from_pretrained(v2_path)
model     = BartForConditionalGeneration.from_pretrained(v2_path).to(device)
print(f"✅ Loaded v2 BART | Device: {device}")

class GuidedDataset(Dataset):
    def __init__(self, df, tok, max_in=256, max_out=128):
        self.inputs  = df['guided_input'].tolist()
        self.targets = df['simple_text'].tolist()
        self.tok = tok
        self.max_in, self.max_out = max_in, max_out

    def __len__(self): return len(self.inputs)

    def __getitem__(self, idx):
        enc = self.tok(self.inputs[idx], max_length=self.max_in,
                       truncation=True, padding='max_length', return_tensors='pt')
        dec = self.tok(text_target=self.targets[idx], max_length=self.max_out,
                       truncation=True, padding='max_length', return_tensors='pt')
        labels = dec['input_ids'].squeeze()
        labels[labels == self.tok.pad_token_id] = -100
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         labels
        }

train_dataset = GuidedDataset(train_df, tokenizer)
val_dataset   = GuidedDataset(val_df,   tokenizer)

training_args = Seq2SeqTrainingArguments(
    output_dir=v3_path,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-6,              # ⬇ lower LR than v2 (was default ~5e-5)
                                     #   prevents overwriting what v2 already learned
    warmup_ratio=0.1,                # gentle warmup — avoids sharp early updates
    weight_decay=0.01,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,
    report_to='none',
    logging_steps=100,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
    # stops after 1 epoch if val loss goes up — key overfitting guard
)

trainer.train()
trainer.save_model(v3_path)
tokenizer.save_pretrained(v3_path)
print(f"✅ v3 BART saved to: {v3_path}")

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Loaded v2 BART | Device: cuda


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 5 — Evaluate SARI on Validation Set
# ============================================================
from easse.sari import corpus_sari

model.eval()

def simplify_batch(texts, batch_size=64):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', max_length=256,
                        truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model.generate(**enc, max_length=128, num_beams=2, do_sample=False)
        all_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return all_preds

predictions = simplify_batch(val_df['guided_input'].tolist())

sari = corpus_sari(
    orig_sents=val_df['complex'].tolist(),
    sys_sents=predictions,
    refs_sents=[val_df['simple_text'].tolist()]
)
print(f"\n✅ v3 SARI Score: {sari:.2f}")
print(f"   Previous (v2): 31.02 | Target: 35.5 | Gap: {35.5 - sari:.2f}")

ModuleNotFoundError: No module named 'easse'

In [ ]:
# ============================================================
# CELL 6 — Generate Final Submission Using RoBERTa Labels
# ============================================================
import json, zipfile

# Load RoBERTa classifier
clf_tokenizer_r = RobertaTokenizerFast.from_pretrained(clf_path)
clf_model_r     = RobertaForSequenceClassification.from_pretrained(clf_path).to(device)
clf_model_r.eval()

def predict_labels(texts, batch_size=256):
    all_labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = clf_tokenizer_r(batch, return_tensors='pt', truncation=True,
                              padding=True, max_length=128).to(device)
        with torch.no_grad():
            preds = clf_model_r(**enc).logits.argmax(dim=-1).cpu().tolist()
        all_labels.extend([id2label[p] for p in preds])
    return all_labels

test_path = '/content/drive/MyDrive/SimpleText2025/simpletext25_task11_test.json'
with open(test_path) as f:
    test_data = json.load(f)

test_sentences     = [item['complex'] for item in test_data]
predicted_labels   = predict_labels(test_sentences)
guided_test_inputs = [f'[{l.upper()}] {s}' for l, s in zip(predicted_labels, test_sentences)]
test_predictions   = simplify_batch(guided_test_inputs)

output = [{
    'pair_id':    item['pair_id'],
    'para_id':    item['para_id'],
    'sent_id':    item['sent_id'],
    'complex':    item['complex'],
    'prediction': pred,
    'run_id':     'tokatrons_task11_BARTPlanGuidedV3'
} for item, pred in zip(test_data, test_predictions)]

out_path = '/content/drive/MyDrive/SimpleText2025/outputs/tokatrons_task11_BARTPlanGuidedV3.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

zip_path = out_path.replace('.json', '.zip')
with zipfile.ZipFile(zip_path, 'w') as zf:
    zf.write(out_path, arcname='tokatrons_task11_BARTPlanGuidedV3.json')

print(f"✅ Submission ready: {zip_path}")
print(f"   Total predictions: {len(output)}")

find whats wrong

In [ ]:
# ============================================================
# STRATEGY: Better Inference on v2 — no retraining needed
# Beam search + length penalty tuned for simplification
# Expected SARI gain: +1 to +3 points over v2's 31.02
# ============================================================
from transformers import BartTokenizer, BartForConditionalGeneration
from easse.sari import corpus_sari
import torch

v2_path   = '/content/drive/MyDrive/SimpleText2025/models/bart-plan-guided'
tokenizer = BartTokenizer.from_pretrained(v2_path)
model     = BartForConditionalGeneration.from_pretrained(v2_path).to(device)
model.eval()

def simplify_batch(texts, batch_size=32, num_beams=4,
                   length_penalty=1.5, min_length=5, no_repeat_ngram_size=3):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', max_length=256,
                        truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_length=128,
                min_length=min_length,
                num_beams=num_beams,
                length_penalty=length_penalty,     # >1 = prefer longer outputs (more content kept)
                no_repeat_ngram_size=no_repeat_ngram_size,  # stops repetition
                early_stopping=True
            )
        all_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return all_preds

# Grid search over inference params on val set — takes ~10 min
import pandas as pd, ast

data_path = '/content/drive/MyDrive/SimpleText2025/cochrane-auto/data'
val_df    = pd.read_csv(f'{data_path}/cochraneauto_sents_val.csv')
val_df.dropna(subset=['complex', 'simple', 'label'], inplace=True)
val_df['simple_text'] = val_df['simple'].apply(
    lambda s: ' '.join(ast.literal_eval(s)) if isinstance(ast.literal_eval(s), list) else str(s)
)
VALID = {'rephrase','delete','split','merge','copy'}
val_df = val_df[val_df['label'].str.strip().str.lower().isin(VALID)].reset_index(drop=True)
val_df['guided_input'] = '[' + val_df['label'].str.upper() + '] ' + val_df['complex']

configs = [
    {'num_beams': 2, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0},  # your current v2 setting
    {'num_beams': 4, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 4, 'length_penalty': 1.5, 'no_repeat_ngram_size': 3},
    {'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 6, 'length_penalty': 1.5, 'no_repeat_ngram_size': 3},
]

best_sari, best_cfg = 0, None
for cfg in configs:
    preds = simplify_batch(val_df['guided_input'].tolist(), **cfg)
    sari  = corpus_sari(
        orig_sents=val_df['complex'].tolist(),
        sys_sents=preds,
        refs_sents=[val_df['simple_text'].tolist()]
    )
    print(f"beams={cfg['num_beams']} lp={cfg['length_penalty']} ngram={cfg['no_repeat_ngram_size']} → SARI: {sari:.2f}")
    if sari > best_sari:
        best_sari, best_cfg = sari, cfg

print(f"\n✅ Best config: {best_cfg}")
print(f"   Best SARI:   {best_sari:.2f}")
print(f"   v2 baseline: 31.02 | Gap to 35.5: {35.5 - best_sari:.2f}")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

beams=2 lp=1.0 ngram=0 → SARI: 35.86
beams=4 lp=1.0 ngram=3 → SARI: 33.30
beams=4 lp=1.5 ngram=3 → SARI: 33.19
beams=4 lp=2.0 ngram=3 → SARI: 33.03


OutOfMemoryError: CUDA out of memory. Tried to allocate 50.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 3.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 13.76 GiB is allocated by PyTorch, and 678.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ============================================================
# Once best config is found — regenerate submission with it
# ============================================================
import json, zipfile, os
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

# Load best label classifier (RoBERTa from Cell 3)
clf_path      = '/content/drive/MyDrive/SimpleText2025/models/roberta-label-clf'
clf_tok       = RobertaTokenizerFast.from_pretrained(clf_path)
clf_model     = RobertaForSequenceClassification.from_pretrained(clf_path).to(device)
clf_model.eval()
id2label      = {0:'rephrase', 1:'delete', 2:'split', 3:'merge', 4:'copy'}

def predict_labels(texts, batch_size=256):
    all_labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = clf_tok(batch, return_tensors='pt', truncation=True,
                      padding=True, max_length=128).to(device)
        with torch.no_grad():
            preds = clf_model(**enc).logits.argmax(dim=-1).cpu().tolist()
        all_labels.extend([id2label[p] for p in preds])
    return all_labels

test_path = '/content/drive/MyDrive/SimpleText2025/simpletext25_task11_test.json'
with open(test_path) as f:
    test_data = json.load(f)

test_sentences   = [item['complex'] for item in test_data]
predicted_labels = predict_labels(test_sentences)
guided_inputs    = [f'[{l.upper()}] {s}' for l, s in zip(predicted_labels, test_sentences)]

# Use the best config found above
test_preds = simplify_batch(guided_inputs, **best_cfg)

output = [{
    'pair_id':    item['pair_id'],
    'para_id':    item['para_id'],
    'sent_id':    item['sent_id'],
    'complex':    item['complex'],
    'prediction': pred,
    'run_id':     'tokatrons_task11_BARTPlanGuidedV2Best'
} for item, pred in zip(test_data, test_preds)]

out_path = '/content/drive/MyDrive/SimpleText2025/outputs/tokatrons_task11_BARTPlanGuidedV2Best.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

zip_path = out_path.replace('.json', '.zip')
with zipfile.ZipFile(zip_path, 'w') as zf:
    zf.write(out_path, arcname='tokatrons_task11_BARTPlanGuidedV2Best.json')

print(f"✅ Submission ready: {zip_path}")

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/SimpleText2025/models/roberta-label-clf'. Use `repo_type` argument if needed.

# ============================================================
# BEST CONFIG FOUND — lock it in
# ============================================================

In [ ]:
best_cfg  = {'num_beams': 2, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0}
best_sari = 35.86

print("✅ TARGET BEATEN!")
print(f"   SARI: 35.86 (target was 35.5)")
print(f"   Config: beams=2, length_penalty=1.0, no_repeat_ngram_size=0")
print(f"   Model:  bart-plan-guided (v2) — no more training needed")

✅ TARGET BEATEN!
   SARI: 35.86 (target was 35.5)
   Config: beams=2, length_penalty=1.0, no_repeat_ngram_size=0
   Model:  bart-plan-guided (v2) — no more training needed


In [ ]:
# ============================================================
# FREE GPU MEMORY
# ============================================================
import torch, gc

# Delete any leftover model objects from the grid search
for var in ['model', 'tokenizer', 'clf_model', 'clf_tok', 'clf_tokenizer']:
    if var in dir():
        del var

gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"GPU memory total: {torch.cuda.mem_get_info()[1]/1e9:.2f} GB")

GPU memory free: 5.57 GB
GPU memory total: 15.64 GB


In [ ]:
# ============================================================
# FINAL SUBMISSION — fixed, with progress tracking
# ============================================================
import json, zipfile, os, torch, gc
from transformers import (BartTokenizer, BartForConditionalGeneration,
                          RobertaTokenizerFast, RobertaForSequenceClassification)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Free any leftover memory first
gc.collect()
torch.cuda.empty_cache()

# Load v2 BART
v2_path   = '/content/drive/MyDrive/SimpleText2025/models/bart-plan-guided'
tokenizer = BartTokenizer.from_pretrained(v2_path)
model     = BartForConditionalGeneration.from_pretrained(v2_path).to(device)
model.eval()

# Load RoBERTa classifier
clf_path  = '/content/drive/MyDrive/SimpleText2025/models/roberta-label-clf'
clf_tok   = RobertaTokenizerFast.from_pretrained(clf_path)
clf_model = RobertaForSequenceClassification.from_pretrained(clf_path).to(device)
clf_model.eval()
id2label  = {0:'rephrase', 1:'delete', 2:'split', 3:'merge', 4:'copy'}

def predict_labels(texts, batch_size=256):
    all_labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = clf_tok(batch, return_tensors='pt', truncation=True,
                      padding=True, max_length=128).to(device)
        with torch.no_grad():
            preds = clf_model(**enc).logits.argmax(dim=-1).cpu().tolist()
        all_labels.extend([id2label[p] for p in preds])
    return all_labels

def simplify_batch(texts, batch_size=64):   # ← 64 instead of 32, faster
    all_preds = []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', max_length=256,
                        truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_length=128,
                num_beams=2,           # hardcoded winning config
                length_penalty=1.0,
                no_repeat_ngram_size=0,
                early_stopping=True
            )
        all_preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
        # Progress print every 10 batches
        if (i // batch_size) % 10 == 0:
            print(f"  Progress: {min(i+batch_size, total)}/{total} sentences done...")
    return all_preds

# Load test data
test_path = '/content/drive/MyDrive/SimpleText2025/simpletext25_task11_test.json'
with open(test_path) as f:
    test_data = json.load(f)

print(f"Test set size: {len(test_data)} sentences")

print("Step 1/3: Predicting labels with RoBERTa...")
test_sentences   = [item['complex'] for item in test_data]
predicted_labels = predict_labels(test_sentences)
print(f"  Done. Label distribution: { {l: predicted_labels.count(l) for l in set(predicted_labels)} }")

print("Step 2/3: Generating simplifications with BART...")
guided_inputs = [f'[{l.upper()}] {s}' for l, s in zip(predicted_labels, test_sentences)]
test_preds    = simplify_batch(guided_inputs)
print("  Done.")

print("Step 3/3: Saving submission file...")
output = [{
    'pair_id':    item['pair_id'],
    'para_id':    item['para_id'],
    'sent_id':    item['sent_id'],
    'complex':    item['complex'],
    'prediction': pred,
    'run_id':     'tokatrons_task11_BARTPlanGuidedFinal'
} for item, pred in zip(test_data, test_preds)]

out_path = '/content/drive/MyDrive/SimpleText2025/outputs/tokatrons_task11_BARTPlanGuidedFinal.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

zip_path = out_path.replace('.json', '.zip')
with zipfile.ZipFile(zip_path, 'w') as zf:
    zf.write(out_path, arcname='tokatrons_task11_BARTPlanGuidedFinal.json')

print(f"\n✅ Final submission ready: {zip_path}")
print(f"   Predictions: {len(output)}")
print(f"   Val SARI: 35.86 | run_id: tokatrons_task11_BARTPlanGuidedFinal")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Test set size: 9160 sentences
Step 1/3: Predicting labels with RoBERTa...
  Done. Label distribution: {'delete': 1976, 'rephrase': 7184}
Step 2/3: Generating simplifications with BART...
  Progress: 64/9160 sentences done...
  Progress: 704/9160 sentences done...
  Progress: 1344/9160 sentences done...
  Progress: 1984/9160 sentences done...
  Progress: 2624/9160 sentences done...
  Progress: 3264/9160 sentences done...
  Progress: 3904/9160 sentences done...
  Progress: 4544/9160 sentences done...
  Progress: 5184/9160 sentences done...
  Progress: 5824/9160 sentences done...
  Progress: 6464/9160 sentences done...
  Progress: 7104/9160 sentences done...
  Progress: 7744/9160 sentences done...
  Progress: 8384/9160 sentences done...
  Progress: 9024/9160 sentences done...
  Done.
Step 3/3: Saving submission file...

✅ Final submission ready: /content/drive/MyDrive/SimpleText2025/outputs/tokatrons_task11_BARTPlanGuidedFinal.zip
   Predictions: 9160
   Val SARI: 35.86 | run_id: tokatron

In [ ]:
# ============================================================
# CELL 1 — Mount + Install
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


!pip install transformers datasets sentencepiece -q
!pip install git+https://github.com/feralvam/easse.git -q

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 93.1 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 2 — Load Data
# ============================================================
import pandas as pd, ast, os, torch

data_path = '/content/drive/MyDrive/SimpleText2025/cochrane-auto/data'
train_df  = pd.read_csv(f'{data_path}/cochraneauto_sents_train.csv')
val_df    = pd.read_csv(f'{data_path}/cochraneauto_sents_val.csv')

def parse_simple(s):
    try:
        lst = ast.literal_eval(s)
        return ' '.join(lst) if isinstance(lst, list) else str(lst)
    except:
        return str(s)

VALID_LABELS = {'rephrase', 'delete', 'split', 'merge', 'copy'}

for df in [train_df, val_df]:
    df.dropna(subset=['complex', 'simple', 'label'], inplace=True)
    df['simple_text']   = df['simple'].apply(parse_simple)
    df['label']         = df['label'].str.strip().str.lower()

train_df = train_df[train_df['label'].isin(VALID_LABELS)].reset_index(drop=True)
val_df   = val_df[val_df['label'].isin(VALID_LABELS)].reset_index(drop=True)

train_df['guided_input'] = '[' + train_df['label'].str.upper() + '] ' + train_df['complex']
val_df['guided_input']   = '[' + val_df['label'].str.upper()   + '] ' + val_df['complex']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Device: {device}")

In [ ]:
# ============================================================
# SETUP — run this first every session
# ============================================================
import torch, gc, os, json, zipfile
from transformers import (BartTokenizer, BartForConditionalGeneration,
                          RobertaTokenizerFast, RobertaForSequenceClassification)
from google.colab import drive
drive.mount('/content/drive')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gc.collect()
torch.cuda.empty_cache()

# Load BART v2
v2_path   = '/content/drive/MyDrive/SimpleText2025/models/bart-plan-guided'
tokenizer = BartTokenizer.from_pretrained(v2_path)
model     = BartForConditionalGeneration.from_pretrained(v2_path).to(device)
model.eval()

# Load RoBERTa classifier
clf_path  = '/content/drive/MyDrive/SimpleText2025/models/roberta-label-clf'
clf_tok   = RobertaTokenizerFast.from_pretrained(clf_path)
clf_model = RobertaForSequenceClassification.from_pretrained(clf_path).to(device)
clf_model.eval()
id2label  = {0:'rephrase', 1:'delete', 2:'split', 3:'merge', 4:'copy'}

def predict_labels(texts, batch_size=256):
    all_labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = clf_tok(batch, return_tensors='pt', truncation=True,
                      padding=True, max_length=128).to(device)
        with torch.no_grad():
            preds = clf_model(**enc).logits.argmax(dim=-1).cpu().tolist()
        all_labels.extend([id2label[p] for p in preds])
    return all_labels

def simplify_batch(texts, batch_size=64):
    all_preds = []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', max_length=256,
                        truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_length=128,
                min_length=3,
                num_beams=2,
                length_penalty=1.0,
                no_repeat_ngram_size=0
            )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        all_preds.extend(decoded)
        if (i // batch_size) % 10 == 0:
            print(f"  {min(i+batch_size, total)}/{total} done...")
    return all_preds

print(f"✅ Setup done | Device: {device}")
print(f"   BART loaded from: {v2_path}")
print(f"   RoBERTa loaded from: {clf_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Setup done | Device: cuda
   BART loaded from: /content/drive/MyDrive/SimpleText2025/models/bart-plan-guided
   RoBERTa loaded from: /content/drive/MyDrive/SimpleText2025/models/roberta-label-clf


In [ ]:
# Run this after Step 1 to get the realistic SARI estimate
# Uses RoBERTa predicted labels on val set (same as what test set will get)
from easse.sari import corpus_sari
import pandas as pd, ast

data_path = '/content/drive/MyDrive/SimpleText2025/cochrane-auto/data'
val_df    = pd.read_csv(f'{data_path}/cochraneauto_sents_val.csv')
val_df.dropna(subset=['complex','simple','label'], inplace=True)
val_df['simple_text'] = val_df['simple'].apply(
    lambda s: ' '.join(ast.literal_eval(s)) if isinstance(ast.literal_eval(s), list) else str(s))
VALID = {'rephrase','delete','split','merge','copy'}
val_df = val_df[val_df['label'].str.strip().str.lower().isin(VALID)].reset_index(drop=True)

# Predict labels with RoBERTa (not oracle)
val_sentences      = val_df['complex'].tolist()
predicted_labels   = predict_labels(val_sentences)
guided_val_inputs  = [f'[{l.upper()}] {s}' for l, s in zip(predicted_labels, val_sentences)]
val_preds          = simplify_batch(guided_val_inputs)

# Compare oracle vs predicted label SARI
oracle_inputs = '[' + val_df['label'].str.upper() + '] ' + val_df['complex']
oracle_preds  = simplify_batch(oracle_inputs.tolist())

oracle_sari = corpus_sari(
    orig_sents=val_df['complex'].tolist(),
    sys_sents=oracle_preds,
    refs_sents=[val_df['simple_text'].tolist()]
)
predicted_sari = corpus_sari(
    orig_sents=val_df['complex'].tolist(),
    sys_sents=val_preds,
    refs_sents=[val_df['simple_text'].tolist()]
)

print(f"SARI with oracle labels (upper bound):    {oracle_sari:.2f}")
print(f"SARI with predicted labels (realistic):   {predicted_sari:.2f}")
print(f"Gap from classifier noise:                {oracle_sari - predicted_sari:.2f}")

  64/1472 done...
  704/1472 done...
  1344/1472 done...
  64/1472 done...
  704/1472 done...
  1344/1472 done...
SARI with oracle labels (upper bound):    35.86
SARI with predicted labels (realistic):   33.23
Gap from classifier noise:                2.63


EVALUATE LABBELLER'S ACCURACY TO IMPROVE

In [ ]:
# ============================================================
# MASTER SETUP CELL — run this first on every session start
# Takes ~4-5 min on T4
# ============================================================

# 1. Mount Drive + Install
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', 'transformers', 'datasets', 'sentencepiece', '-q'])
subprocess.run(['pip', 'install', 'git+https://github.com/feralvam/easse.git', '-q'])

# 2. Imports
import torch, gc, os, json, zipfile, ast
import pandas as pd
import numpy as np
from transformers import (
    BartTokenizer, BartForConditionalGeneration,
    RobertaTokenizerFast, RobertaForSequenceClassification,
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    Seq2SeqTrainer, Seq2SeqTrainingArguments
)
from datasets import Dataset as HFDataset
from torch.utils.data import Dataset
from easse.sari import corpus_sari

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gc.collect()
torch.cuda.empty_cache()
print(f"✅ Imports done | Device: {device}")

# 3. Load Data
data_path = '/content/drive/MyDrive/SimpleText2025/cochrane-auto/data'

train_df = pd.read_csv(f'{data_path}/cochraneauto_sents_train.csv')
val_df   = pd.read_csv(f'{data_path}/cochraneauto_sents_val.csv')

def parse_simple(s):
    try:
        lst = ast.literal_eval(s)
        return ' '.join(lst) if isinstance(lst, list) else str(lst)
    except:
        return str(s)

VALID_LABELS = {'rephrase', 'delete', 'split', 'merge', 'copy'}
for df in [train_df, val_df]:
    df.dropna(subset=['complex', 'simple', 'label'], inplace=True)
    df['simple_text'] = df['simple'].apply(parse_simple)
    df['label']       = df['label'].str.strip().str.lower()

train_df = train_df[train_df['label'].isin(VALID_LABELS)].reset_index(drop=True)
val_df   = val_df[val_df['label'].isin(VALID_LABELS)].reset_index(drop=True)

train_df['guided_input'] = '[' + train_df['label'].str.upper() + '] ' + train_df['complex']
val_df['guided_input']   = '[' + val_df['label'].str.upper()   + '] ' + val_df['complex']

label2id = {'rephrase': 0, 'delete': 1, 'split': 2, 'merge': 3, 'copy': 4}
id2label  = {v: k for k, v in label2id.items()}

print(f"✅ Data loaded | Train: {len(train_df)} | Val: {len(val_df)}")

# 4. Load BART v2
v2_path   = '/content/drive/MyDrive/SimpleText2025/models/bart-plan-guided'
tokenizer = BartTokenizer.from_pretrained(v2_path)
model     = BartForConditionalGeneration.from_pretrained(v2_path).to(device)
model.eval()
print(f"✅ BART v2 loaded from: {v2_path}")

# 5. Load RoBERTa classifier
clf_path      = '/content/drive/MyDrive/SimpleText2025/models/roberta-label-clf'
clf_tok       = RobertaTokenizerFast.from_pretrained(clf_path)
clf_model     = RobertaForSequenceClassification.from_pretrained(clf_path).to(device)
clf_model.eval()
print(f"✅ RoBERTa classifier loaded from: {clf_path}")

# 6. Core Functions
def simplify_batch(texts, batch_size=64):
    all_preds = []
    total = len(texts)
    for i in range(0, total, batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', max_length=256,
                        truncation=True, padding=True).to(device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_length=128,
                min_length=3,
                num_beams=2,
                length_penalty=1.0,
                no_repeat_ngram_size=0
            )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        all_preds.extend(decoded)
        if (i // batch_size) % 10 == 0:
            print(f"  {min(i+batch_size, total)}/{total} done...")
    return all_preds

def predict_labels(texts, batch_size=256):
    all_labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = clf_tok(batch, return_tensors='pt', truncation=True,
                      padding=True, max_length=128).to(device)
        with torch.no_grad():
            preds = clf_model(**enc).logits.argmax(dim=-1).cpu().tolist()
        all_labels.extend([id2label[p] for p in preds])
    return all_labels

def eval_sari(preds, use_oracle=False):
    refs = val_df['simple_text'].tolist()
    return corpus_sari(
        orig_sents=val_df['complex'].tolist(),
        sys_sents=preds,
        refs_sents=[refs]
    )

def generate_submission(predictions, run_id, test_data):
    out_path = f'/content/drive/MyDrive/SimpleText2025/outputs/{run_id}.json'
    zip_path = out_path.replace('.json', '.zip')
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    output = [{
        'pair_id':    item['pair_id'],
        'para_id':    item['para_id'],
        'sent_id':    item['sent_id'],
        'complex':    item['complex'],
        'prediction': pred,
        'run_id':     run_id
    } for item, pred in zip(test_data, predictions)]
    with open(out_path, 'w') as f:
        json.dump(output, f, indent=2)
    with zipfile.ZipFile(zip_path, 'w') as zf:
        zf.write(out_path, arcname=f'{run_id}.json')
    empty = sum(1 for p in predictions if p.strip() == "")
    print(f"✅ Saved: {zip_path}")
    print(f"   Total: {len(output)} | Empty: {empty}")
    return zip_path

# 7. Load test data
test_path = '/content/drive/MyDrive/SimpleText2025/simpletext25_task11_test.json'
with open(test_path) as f:
    test_data = json.load(f)
print(f"✅ Test data loaded | {len(test_data)} sentences")

# 8. Load DeBERTa if already trained
deberta_clf_path = '/content/drive/MyDrive/SimpleText2025/models/deberta-label-clf'
if os.path.exists(deberta_clf_path):
    clf_tokenizer2 = AutoTokenizer.from_pretrained(deberta_clf_path)
    clf_model2     = AutoModelForSequenceClassification.from_pretrained(deberta_clf_path).to(device)
    clf_model2.eval()

    def predict_labels_deberta(texts, batch_size=128):
        all_labels = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = clf_tokenizer2(batch, return_tensors='pt', truncation=True,
                                 padding=True, max_length=128).to(device)
            with torch.no_grad():
                preds = clf_model2(**enc).logits.argmax(dim=-1).cpu().tolist()
            all_labels.extend([id2label[p] for p in preds])
        return all_labels

    print(f"✅ DeBERTa classifier loaded from: {deberta_clf_path}")
else:
    print("ℹ️  DeBERTa not trained yet — run the DeBERTa training cell to train it")

# ── SUMMARY ──────────────────────────────────────────
print("\n" + "="*50)
print("SESSION READY — what's available:")
print(f"  model            → BART v2 (SARI 35.86 oracle / 33.23 realistic)")
print(f"  tokenizer        → BART tokenizer")
print(f"  clf_model        → RoBERTa label classifier")
print(f"  clf_model2       → DeBERTa classifier ({'✅ loaded' if os.path.exists(deberta_clf_path) else '⬜ not trained yet'})")
print(f"  train_df         → {len(train_df)} rows")
print(f"  val_df           → {len(val_df)} rows")
print(f"  test_data        → {len(test_data)} sentences")
print(f"  simplify_batch() → generate simplifications")
print(f"  predict_labels() → RoBERTa label prediction")
print(f"  predict_labels_deberta() → DeBERTa label prediction ({'✅ ready' if os.path.exists(deberta_clf_path) else '⬜ not ready'})")
print(f"  eval_sari()      → compute SARI on val set")
print(f"  generate_submission() → save + zip submission file")
print("="*50)

Mounted at /content/drive
✅ Imports done | Device: cuda
✅ Data loaded | Train: 10188 | Val: 1472


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

✅ BART v2 loaded from: /content/drive/MyDrive/SimpleText2025/models/bart-plan-guided


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ RoBERTa classifier loaded from: /content/drive/MyDrive/SimpleText2025/models/roberta-label-clf
✅ Test data loaded | 9160 sentences
ℹ️  DeBERTa not trained yet — run the DeBERTa training cell to train it

SESSION READY — what's available:
  model            → BART v2 (SARI 35.86 oracle / 33.23 realistic)
  tokenizer        → BART tokenizer
  clf_model        → RoBERTa label classifier
  clf_model2       → DeBERTa classifier (⬜ not trained yet)
  train_df         → 10188 rows
  val_df           → 1472 rows
  test_data        → 9160 sentences
  simplify_batch() → generate simplifications
  predict_labels() → RoBERTa label prediction
  predict_labels_deberta() → DeBERTa label prediction (⬜ not ready)
  eval_sari()      → compute SARI on val set
  generate_submission() → save + zip submission file


In [ ]:
# ============================================================
# STEP 1 — Check RoBERTa classifier accuracy first
# This tells us how much room we have to improve
# ============================================================
import numpy as np

val_true      = val_df['label'].str.strip().str.lower().tolist()
val_predicted = predict_labels(val_df['complex'].tolist())

correct = sum(t == p for t, p in zip(val_true, val_predicted))
total   = len(val_true)
print(f"Classifier accuracy: {correct}/{total} = {correct/total*100:.1f}%")

# Per-label breakdown
from collections import Counter
label_correct = Counter()
label_total   = Counter()
for t, p in zip(val_true, val_predicted):
    label_total[t] += 1
    if t == p:
        label_correct[t] += 1

print("\nPer-label accuracy:")
for lbl in ['rephrase', 'delete', 'split', 'merge', 'copy']:
    acc = label_correct[lbl] / label_total[lbl] * 100 if label_total[lbl] > 0 else 0
    print(f"  {lbl:10s}: {label_correct[lbl]:4d}/{label_total[lbl]:4d} = {acc:.1f}%")

Classifier accuracy: 765/1472 = 52.0%

Per-label accuracy:
  rephrase  :  611/ 758 = 80.6%
  delete    :  154/ 592 = 26.0%
  split     :    0/  58 = 0.0%
  merge     :    0/  64 = 0.0%
  copy      :    0/   0 = 0.0%


In [ ]:
# ============================================================
# DEBERTA CLASSIFIER TRAINING — fixed
# ============================================================
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from datasets import Dataset as HFDataset
import numpy as np

label2id = {'rephrase': 0, 'delete': 1, 'split': 2, 'merge': 3, 'copy': 4}
id2label  = {v: k for k, v in label2id.items()}

model_name     = 'microsoft/deberta-v3-base'
clf_tokenizer2 = AutoTokenizer.from_pretrained(model_name)
clf_model2     = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=5, id2label=id2label, label2id=label2id
)

def tokenize(batch):
    return clf_tokenizer2(batch['text'], truncation=True,
                          padding='max_length', max_length=128)

train_data = HFDataset.from_dict({
    'text':  train_df['complex'].tolist(),
    'label': [label2id[l] for l in train_df['label']]
}).map(tokenize, batched=True)

val_data = HFDataset.from_dict({
    'text':  val_df['complex'].tolist(),
    'label': [label2id[l] for l in val_df['label']]
}).map(tokenize, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': (preds == labels).mean()}

clf_args2 = TrainingArguments(
    output_dir='/content/deberta_clf',
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=200,               # ✅ fixed: was warmup_ratio (deprecated)
    weight_decay=0.01,
    label_smoothing_factor=0.1,
    fp16=False,                     # ✅ fixed: DeBERTa-v3 conflicts with fp16
    bf16=False,                     # keep both off — T4 doesn't support bf16 either
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none',
)

clf_trainer2 = Trainer(
    model=clf_model2,
    args=clf_args2,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

clf_trainer2.train()

deberta_clf_path = '/content/drive/MyDrive/SimpleText2025/models/deberta-label-clf'
clf_model2.save_pretrained(deberta_clf_path)
clf_tokenizer2.save_pretrained(deberta_clf_path)
print("✅ DeBERTa classifier saved!")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

Map:   0%|          | 0/10188 [00:00<?, ? examples/s]

Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,1.169008,1.134157,0.514946
2,1.140053,nan,0.514946
3,1.133121,nan,0.514946


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ DeBERTa classifier saved!


In [ ]:
# ============================================================
# STEP 3 — Evaluate new classifier vs old on val set
# ============================================================
from easse.sari import corpus_sari

# Update predict_labels to use DeBERTa
def predict_labels_deberta(texts, batch_size=128):
    all_labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = clf_tokenizer2(batch, return_tensors='pt', truncation=True,
                             padding=True, max_length=128).to(device)
        with torch.no_grad():
            preds = clf_model2(**enc).logits.argmax(dim=-1).cpu().tolist()
        all_labels.extend([id2label[p] for p in preds])
    return all_labels

# Check accuracy
val_predicted_deb = predict_labels_deberta(val_df['complex'].tolist())
val_true          = val_df['label'].str.strip().str.lower().tolist()
acc = sum(t==p for t,p in zip(val_true, val_predicted_deb)) / len(val_true)
print(f"DeBERTa accuracy: {acc*100:.1f}%  (RoBERTa was ??%)")

# Check SARI with DeBERTa labels
guided_deb = [f'[{l.upper()}] {s}' for l,s in zip(val_predicted_deb, val_df['complex'].tolist())]
preds_deb  = simplify_batch(guided_deb)

sari_deb = corpus_sari(
    orig_sents=val_df['complex'].tolist(),
    sys_sents=preds_deb,
    refs_sents=[val_df['simple_text'].tolist()]
)
print(f"\nSARI with RoBERTa labels: 33.23")
print(f"SARI with DeBERTa labels: {sari_deb:.2f}")
print(f"Improvement:              +{sari_deb - 33.23:.2f}")
print(f"Gap to target (35.5):     {35.5 - sari_deb:.2f}")

DeBERTa accuracy: 51.5%  (RoBERTa was ??%)
  64/1472 done...
  704/1472 done...
  1344/1472 done...

SARI with RoBERTa labels: 33.23
SARI with DeBERTa labels: 33.85
Improvement:              +0.62
Gap to target (35.5):     1.65


In [ ]:
# ============================================================
# RoBERTa v2 — BALANCED weights, not too aggressive
# ============================================================
from transformers import (RobertaTokenizerFast, RobertaForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from datasets import Dataset as HFDataset
from torch import nn
from collections import Counter
import torch, numpy as np, pandas as pd

label2id = {'rephrase': 0, 'delete': 1, 'split': 2, 'merge': 3, 'copy': 4}
id2label  = {v: k for k, v in label2id.items()}

# ── Milder oversampling — 2x instead of 5x ──
minority       = train_df[train_df['label'].isin(['split', 'merge'])]
train_balanced = pd.concat(
    [train_df] + [minority]*2, ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced train label counts:")
print(train_balanced['label'].value_counts())

# ── Softer class weights — cap at 2.5x max ──
label_counts = Counter(train_df['label'].tolist())
total        = sum(label_counts.values())
raw_weights  = {
    id2label[i]: total / (5 * label_counts[id2label[i]]) if id2label[i] in label_counts and label_counts[id2label[i]] > 0
    else 1.0
    for i in range(5)
}
# Cap weights so no label gets more than 2.5x the minimum weight
min_w = min(raw_weights.values())
capped = {k: min(v, min_w * 2.5) for k, v in raw_weights.items()}
class_weights_cpu = torch.tensor(
    [capped[id2label[i]] for i in range(5)], dtype=torch.float32
)
print("Raw weights:    ", {k: f"{v:.2f}" for k, v in raw_weights.items()})
print("Capped weights: ", {k: f"{v:.2f}" for k, v in capped.items()})

# ── Model ──
roberta_name   = 'roberta-base'
clf_tokenizer5 = RobertaTokenizerFast.from_pretrained(roberta_name)
clf_model5     = RobertaForSequenceClassification.from_pretrained(
    roberta_name, num_labels=5, id2label=id2label, label2id=label2id
)

def tokenize(batch):
    return clf_tokenizer5(batch['text'], truncation=True,
                          padding='max_length', max_length=128)

train_data = HFDataset.from_dict({
    'text':  train_balanced['complex'].tolist(),
    'label': [label2id[l] for l in train_balanced['label']]
}).map(tokenize, batched=True)

val_data = HFDataset.from_dict({
    'text':  val_df['complex'].tolist(),
    'label': [label2id[l] for l in val_df['label']]
}).map(tokenize, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = (preds == labels).mean()
    per_class = {}
    for i, name in id2label.items():
        mask = labels == i
        if mask.sum() > 0:
            per_class[f'acc_{name}'] = float((preds[mask] == labels[mask]).mean())
    return {'accuracy': acc, **per_class}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        w       = class_weights_cpu.to(device=outputs.logits.device,
                                       dtype=outputs.logits.dtype)
        loss    = nn.CrossEntropyLoss(weight=w)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

clf_args5 = TrainingArguments(
    output_dir='/content/roberta_balanced',
    num_train_epochs=6,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none',
)

clf_trainer5 = WeightedTrainer(
    model=clf_model5,
    args=clf_args5,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\nTraining balanced RoBERTa...")
clf_trainer5.train()

roberta_b_path = '/content/drive/MyDrive/SimpleText2025/models/roberta-balanced-clf'
clf_model5.save_pretrained(roberta_b_path)
clf_tokenizer5.save_pretrained(roberta_b_path)
print("✅ Balanced RoBERTa saved!")

Balanced train label counts:
label
rephrase    5239
delete      4063
split       1563
merge       1095
Name: count, dtype: int64
Raw weights:     {'rephrase': '0.39', 'delete': '0.50', 'split': '3.91', 'merge': '5.58', 'copy': '1.00'}
Capped weights:  {'rephrase': '0.39', 'delete': '0.50', 'split': '0.97', 'merge': '0.97', 'copy': '0.97'}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/11960 [00:00<?, ? examples/s]

Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Balanced RoBERTa saved!


In [ ]:
# ============================================================
# EVALUATE
# ============================================================
from easse.sari import corpus_sari

clf_model5.eval()

def predict_labels_balanced(texts, batch_size=256):
    all_labels = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = clf_tokenizer5(batch, return_tensors='pt', truncation=True,
                               padding=True, max_length=128).to(device)
        with torch.no_grad():
            preds = clf_model5(**enc).logits.argmax(dim=-1).cpu().tolist()
        all_labels.extend([id2label[p] for p in preds])
    return all_labels

val_pred_b = predict_labels_balanced(val_df['complex'].tolist())
val_true   = val_df['label'].str.strip().str.lower().tolist()

print("Per-label accuracy:")
print(f"{'Label':10s} | {'Count':>6} | {'Original':>9} | {'Weighted':>9} | {'Balanced':>9}")
print("-" * 55)
for lbl in ['rephrase', 'delete', 'split', 'merge']:
    total = sum(1 for t in val_true if t == lbl)
    if total == 0: continue
    orig_c = sum(1 for t,p in zip(val_true, val_pred_roberta)  if t==lbl and p==lbl)
    wt_c   = sum(1 for t,p in zip(val_true, val_pred_rw)       if t==lbl and p==lbl)
    bal_c  = sum(1 for t,p in zip(val_true, val_pred_b)        if t==lbl and p==lbl)
    print(f"{lbl:10s} | {total:>6} | {orig_c/total*100:>8.1f}% | {wt_c/total*100:>8.1f}% | {bal_c/total*100:>8.1f}%")

overall_b = sum(t==p for t,p in zip(val_true, val_pred_b)) / len(val_true)
print(f"\nOverall accuracy: {overall_b*100:.1f}%")

guided_b = [f'[{l.upper()}] {s}' for l,s in zip(val_pred_b, val_df['complex'].tolist())]
preds_b  = simplify_batch(guided_b)
sari_b   = corpus_sari(
    orig_sents=val_df['complex'].tolist(),
    sys_sents=preds_b,
    refs_sents=[val_df['simple_text'].tolist()]
)
print(f"\nSARI progression:")
print(f"  RoBERTa original:  33.23")
print(f"  RoBERTa weighted:  33.55")
print(f"  RoBERTa balanced:  {sari_b:.2f}")
print(f"  Oracle upper bound: 35.86")
print(f"  Gap to 35.5:        {35.5 - sari_b:.2f}")

Per-label accuracy:
Label      |  Count |  Original |  Weighted |  Balanced
-------------------------------------------------------
rephrase   |    758 |     80.6% |      0.0% |     98.0%
delete     |    592 |     26.0% |      0.0% |      2.9%
split      |     58 |      0.0% |     50.0% |      3.4%
merge      |     64 |      0.0% |     65.6% |      0.0%

Overall accuracy: 51.8%
  64/1472 done...
  704/1472 done...
  1344/1472 done...

SARI progression:
  RoBERTa original:  33.23
  RoBERTa weighted:  33.55
  RoBERTa balanced:  33.83
  Oracle upper bound: 35.86
  Gap to 35.5:        1.67
